## Objectives
- Understand differences between `threading` and `multiprocessing` in Python
- Write a script using `ThreadPoolExecutor` to parallelize I/O-bound tasks
- Explain the Global Interpreter Lock (GIL) and its implications
- Describe when to use `asyncio` vs `threading` vs `multiprocessing`

## 1) Threading vs Multiprocessing — core idea

- `threading`: multiple threads share the same memory (same process). Low-cost context switches, ideal for I/O-bound tasks where threads spend time waiting (network, disk, sleep).
- `multiprocessing`: multiple processes each have their own Python interpreter and memory space. Higher overhead (IPC, memory) but bypasses the GIL, making it suitable for CPU-bound tasks.

Two trade-offs to remember: memory isolation (processes) vs lightweight concurrency (threads).

In [2]:
import time
from threading import Thread

def io_task(i):
    print(f"Task {i} started")
    time.sleep(1)  # Simulate I/O operation
    print(f"Task {i} finished")

# ---------------- Sequential ----------------
start = time.perf_counter()

for i in range(3):
    io_task(i)

end = time.perf_counter()
print(f"\nSequential time: {end - start:.2f} seconds")

# ---------------- Threading ----------------
start = time.perf_counter()

threads = [Thread(target=io_task, args=(i,)) for i in range(3)]

for t in threads:
    t.start()

for t in threads:
    t.join()

end = time.perf_counter()
print(f"\nThreaded time: {end - start:.2f} seconds")

Task 0 started
Task 0 finished
Task 1 started
Task 1 finished
Task 2 started
Task 2 finished

Sequential time: 3.00 seconds
Task 0 started
Task 1 started
Task 2 started
Task 0 finishedTask 2 finished
Task 1 finished


Threaded time: 1.02 seconds


## 2) `ThreadPoolExecutor` for I/O-bound tasks

Use `concurrent.futures.ThreadPoolExecutor` to simplify thread pools. Good for many short I/O tasks (HTTP calls, file reads). Always avoid creating an excessive number of threads.

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def simulated_io(i):
    print(f"Task {i} started")
    time.sleep(1)          # Simulate I/O operation
    print(f"Task {i} finished")
    return f"Result {i}"

start = time.perf_counter()

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = [
        executor.submit(simulated_io, i)
        for i in range(6)
    ]

    for future in as_completed(futures):
        print(future.result())

end = time.perf_counter()

print(f"\nExecution Time: {end - start:.2f} seconds")

Task 0 started
Task 1 started
Task 2 started
Task 0 finished
Task 3 started
Result 0
Task 1 finished
Task 4 started
Result 1
Task 2 finished
Task 5 started
Result 2
Task 3 finishedTask 4 finished

Result 4
Result 3
Task 5 finished
Result 5

Execution Time: 2.01 seconds


## 3) The Global Interpreter Lock (GIL)

- The GIL is a mutex that protects access to Python objects, preventing multiple native threads from executing Python bytecodes at once in a single process.
- Effect: CPU-bound Python code (pure Python loops, heavy math in Python) cannot fully utilize multiple cores with `threading`. Threads will compete for the GIL, causing limited CPU speedup.
- Workarounds: use `multiprocessing` (separate interpreters, no shared GIL), write CPU-heavy parts in C extensions or use libraries that release the GIL (e.g., NumPy, many I/O libraries).

Bottom line: `threading` is excellent for I/O-bound tasks; `multiprocessing` or native extensions are needed for CPU-bound parallelism in CPython.

## 4) `multiprocessing` / `ProcessPoolExecutor` for CPU-bound tasks

Use separate processes to run CPU-heavy work in parallel across cores. `concurrent.futures.ProcessPoolExecutor` is a convenient high-level API.

In [4]:
# Example 3: ProcessPoolExecutor for CPU-bound work
from concurrent.futures import ProcessPoolExecutor, as_completed
import math, time

def cpu_task(n):
    # some CPU heavy work: sum of sqrt up to n
    s = 0.0
    for i in range(1, n):
        s += math.sqrt(i)
    return s

# Compare serial vs process pool for moderately large n
nums = [2000000, 2000000]
start = time.perf_counter()
results = [cpu_task(n) for n in nums]
end = time.perf_counter()
print('Serial CPU time:', end - start)

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as ex:
    futures = [ex.submit(cpu_task, n) for n in nums]
    for fut in as_completed(futures):
        print('proc result done')
end = time.perf_counter()
print('ProcessPool time:', end - start)


Serial CPU time: 0.3101827000000412
proc result done
proc result done
ProcessPool time: 0.16069300000071962


## 5) `asyncio` vs `threading` vs `multiprocessing` — when to choose

- Use `asyncio` when: you have many concurrent I/O tasks that are cooperative (libraries are async-friendly), you want low-overhead concurrency without threads, and you can use async libraries (e.g., `aiohttp`).
- Use `threading` when: tasks are I/O-bound and blocking libraries are used (or when mixing with blocking code is simpler). Threads are easy to use and integrate with existing blocking APIs.
- Use `multiprocessing` when: tasks are CPU-bound and you need to use multiple cores; when you must bypass the GIL for pure Python CPU work.

Practical rule: prefer `asyncio` for high-concurrency I/O with async libraries; `ThreadPoolExecutor` for simple parallel I/O with blocking libraries; `ProcessPoolExecutor` for CPU-heavy work.

In [ ]:
import asyncio
import time

async def async_io(i):
    print(f"Task {i} started")
    await asyncio.sleep(1)      # Simulate I/O wait
    print(f"Task {i} finished")
    return f"Result {i}"

async def main():

    tasks = [
        async_io(i)
        for i in range(6)
    ]

    results = await asyncio.gather(*tasks)

    print("\nResults:", results)

start = time.perf_counter()

asyncio.run(main())

end = time.perf_counter()

print(f"\nExecution Time: {end - start:.2f} seconds")

## 6) Quick checklist

- I/O-bound, blocking libraries -> `ThreadPoolExecutor` or `threading`
- I/O-bound, async libraries -> `asyncio`
- CPU-bound (pure Python) -> `multiprocessing` / `ProcessPoolExecutor`
- CPU-bound but using libraries that release GIL (NumPy) -> consider library parallelism

## 7) References and next steps
- `concurrent.futures` documentation: https://docs.python.org/3/library/concurrent.futures.html
- `asyncio` documentation: https://docs.python.org/3/library/asyncio.html

Try the examples locally; adjust `max_workers` and problem sizes to observe scaling on your machine.

## Practical: benchmarking HTTP fetch — sequential vs ThreadPoolExecutor

This example fetches multiple URLs and measures time for sequential requests vs parallel requests using `ThreadPoolExecutor`.
Observations: network latency dominates per-request time; parallelism reduces wall-clock time for I/O-bound workloads. Requires the `requests` package and network access.

In [10]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

urls = [
    "https://www.python.org",
    "https://httpbin.org/delay/1",
    "https://www.google.com",
    "https://www.github.com",
    "https://www.stackoverflow.com",
    "https://www.wikipedia.org",
]

def fetch(url, timeout=10):
    try:
        response = requests.get(url, timeout=timeout)
        return url, response.status_code, len(response.content)
    except Exception:
        return url, None, 0

# ---------------- Sequential ----------------
start = time.perf_counter()

seq_results = []

for url in urls:
    seq_results.append(fetch(url))

seq_time = time.perf_counter() - start

# ---------------- ThreadPoolExecutor ----------------
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=min(10, len(urls))) as executor:

    futures = [
        executor.submit(fetch, url)
        for url in urls
    ]

    par_results = []

    for future in as_completed(futures):
        par_results.append(future.result())

par_time = time.perf_counter() - start

# ---------------- Results ----------------
print(f"\nSequential Time : {seq_time:.2f} seconds")
print(f"Parallel Time   : {par_time:.2f} seconds")

if par_time > 0:
    print(f"Speedup         : {seq_time / par_time:.2f}x")

print("\nSequential Results:")
for result in seq_results:
    print(result)

print("\nParallel Results (unordered):")
for result in par_results:
    print(result)


Sequential Time : 24.85 seconds
Parallel Time   : 11.52 seconds
Speedup         : 2.16x

Sequential Results:
('https://www.python.org', 200, 53036)
('https://httpbin.org/delay/1', 200, 361)
('https://www.google.com', 200, 81305)
('https://www.github.com', 200, 578152)
('https://www.stackoverflow.com', 200, 488692)
('https://www.wikipedia.org', 403, 126)

Parallel Results (unordered):
('https://www.wikipedia.org', 403, 126)
('https://www.python.org', 200, 53036)
('https://www.github.com', 200, 578152)
('https://www.stackoverflow.com', 200, 488407)
('https://httpbin.org/delay/1', 200, 361)
('https://www.google.com', 200, 81123)
